In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore") # umap deprecation warnings related to numba do not concern us.


# GRN Construction

Builds the prior knowledge network (PKN) from CollecTRI, using the per-phenotype binarized observation data (`meta`) produced by **"4.5. Phenotype Observations.ipynb"**.

This notebook ends by exporting `{FILE_PREFIX}_prior_grn.json`, `{FILE_PREFIX}_pkn.json`, `{FILE_PREFIX}_data_meta.json` and `{FILE_PREFIX}_config.json`.

The actual BoNesis solving step lives in **"6. BN Inference (BoNesis).ipynb"**, which consumes these exported files. Set the same `FILE_PREFIX` there.

In [ ]:
import sys
import json
from pathlib import Path as path

import numpy as np
import pandas as pd

from markers import (
    states_and_markers, # Dictionnary containing phenotypes and their marker genes
    marker_genes, # pandas.Series containing simply the name of markers
)

# Repo root → src on path (shared with the RNA velocity / trajectory-inference notebooks)
ROOT = path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from grn_inference import (
    load_collectri,
    in_db,
    df_to_bonesis_data,
    export_pkn_bundle,
    df_to_graph,
    extract_largest_scc,
    extract_largest_wcc,
)


In [ ]:
FILE_PREFIX = "sum"  # must match FILE_PREFIX used in "4.5. Phenotype Observations.ipynb"
PKN_SOURCE = "biggest_wcc"  # "biggest_wcc" | "prior_grn"

print(f"FILE_PREFIX={FILE_PREFIX!r}, PKN_SOURCE={PKN_SOURCE!r}")


### TF database loading

In [ ]:
net = load_collectri()


In [ ]:
meta = pd.read_csv(f"meta_{FILE_PREFIX}_bin_aggregated.csv", index_col=0)
with open(f"{FILE_PREFIX}_config.json") as _cfg:
    INCLUDE_HC_CONSTRAINT = json.load(_cfg)["INCLUDE_HC_CONSTRAINT"]

meta_var = meta
print(meta.shape, f"INCLUDE_HC_CONSTRAINT={INCLUDE_HC_CONSTRAINT}")
meta.iloc[:, :10]


In [ ]:
null_var_mask = meta.var(skipna=False) == 0
null_var_genes = meta.columns[null_var_mask]
null_var_genes.shape


In [ ]:
in_db(meta_var, net).mean()


In [ ]:
in_db(meta_var[marker_genes], net).mean()


In [ ]:
meta_var[null_var_genes].columns[~in_db(meta_var[null_var_genes], net)]


In [ ]:
print(meta_var.shape)
meta_var_db = meta_var[meta_var.columns[in_db(meta_var, net)]]
print(meta_var_db.shape)
meta_var_db.iloc[:, :10]


In [ ]:
meta_var_db.T.isna().mean()


In [ ]:
meta_var_db.isna().mean().mean()


In [ ]:
import networkx as nx


In [ ]:
# The whole GRN built from the full (unfiltered) CollecTRI network
whole_grn = df_to_graph(net, source="source", target="target")
induced_grn = nx.induced_subgraph(whole_grn, meta_var_db.columns)
len(whole_grn), len(induced_grn)


In [ ]:
core_tf_nw = extract_largest_scc(whole_grn)
len(core_tf_nw)


In [ ]:
prior_grn = (
    nx.induced_subgraph(
        whole_grn,
        set(core_tf_nw).union(
            meta_var_db.columns
        )
    )
)
len(prior_grn)


In [ ]:
nx.is_weakly_connected(prior_grn)


In [ ]:
pkn_biggest_wcc = extract_largest_wcc(prior_grn)
len(pkn_biggest_wcc)


In [ ]:
all(i in pkn_biggest_wcc for i in core_tf_nw)


In [ ]:
marker_genes.isin(prior_grn).mean()


In [ ]:
marker_genes.isin(pkn_biggest_wcc).mean()


In [ ]:
print("\t".join(["Cell", "pTotal", "pBiggestWCC"]))
for pheno, genes in states_and_markers.items():
    if pheno in meta.index:
        print("\t".join(
            (lambda __: f"{100*__:.2f}" if isinstance(__, float) else __)(_)
            for _ in
            [pheno, pd.Index(genes).isin(whole_grn).mean(), pd.Index(genes).isin(pkn_biggest_wcc).mean()]
        ))


In [ ]:
import bonesis


In [ ]:
data_meta = df_to_bonesis_data(meta_var_db)
data_meta.keys()


In [ ]:
if PKN_SOURCE == "prior_grn":
    _pkn_graph = prior_grn
elif PKN_SOURCE == "biggest_wcc":
    _pkn_graph = pkn_biggest_wcc
else:
    raise ValueError(PKN_SOURCE)

pkn = bonesis.domains.InfluenceGraph(
    _pkn_graph, maxclause=8, allow_skipping_nodes=True, canonic=False
)
print(f"PKN_SOURCE={PKN_SOURCE!r}, nodes={len(_pkn_graph)}")


In [ ]:
bo = bonesis.BoNesis(pkn, data_meta)
bo


In [ ]:
export_pkn_bundle(FILE_PREFIX, prior_grn, pkn, data_meta, INCLUDE_HC_CONSTRAINT)

print(
    f"Exported prior_grn ({len(prior_grn)} nodes), pkn ({len(pkn)} nodes), "
    f"data_meta ({len(data_meta)} observations) and config for FILE_PREFIX={FILE_PREFIX!r}"
)


### Next step

Continue in **"6. BN Inference (BoNesis).ipynb"** — set `FILE_PREFIX` there to the same value as `FILE_PREFIX` above, to load the files exported here.